##AUTOENCODER SCRIPT = this will perform a dimensionality reduction on both GH administed control data and all athlete data
ensure to activate uv environment in powershell: .\.venv\Scripts\activate.ps1

In [ ]:
#Autoencoder works on all data GHadmin. data and the athlete data itself = merged_df 

import tensorflow as tf
from tensorflow.keras import Dense, layers, models
import pandas as pd
from config import data as dcfg
import pandas as pd
from config import model as mcfg

#import matplotlib.pyplot as plt


Import Data ising panda 
Import Config class to reference the path to the cleaned GH data
The autoencoder can not process non-numerical values - therefore they need to be removed and stored in another variable: ids (identies)

In [21]:
df = pd.read_csv(dcfg.gh_admin)
df.shape 
df.tail(7)
#99*5 = 495 this is the flattened vector that goes into the autoencoder?
ids = df.iloc[:,0]
numeric_df = df.drop(columns=["Volunteer"])



now that the dataset has been read in, and cleaned from previous R work, The autoencoder needs an input:
This is going to be a tabular outlier autoencoder 

In [ ]:
#convert data to tensor - tensorflow NN needs to be able to read the inptu data so it should be in the tensor format 
    # to ensure all data is numerical we use .numpy to avoid errors
df_tensor = tf.convert_to_tensor(numeric_df, dtype = tf.float32).numpy()
#shuffled_points = tf.random.shuffle(df_tensor)

#Do you think the volunteer name will throw an error?

Autoencoder - the input must always match the output - it is essentially recreating the dataset with its own found knowledge of what is normal vs what isn't

1. Input: 4 (Sex, Age, PNP, IGF)

2. Bottleneck: 3 (Your "Visualization" coordinates)

3. Output: 4 (Reconstructed Sex, Age, PNP, IGF)

Our bottleneck vector will be  TUPLE - this means that it is enclosed in ROUND brackets - it can not be changed once made
    - Different from a list which is enclosed in SQUARE brackets and is able to be modified.


In [ ]:

def autoenc_build(input_shape):
    model = models.Sequential([
        # this is the encoder part
        layers.Input(input_shape = (input_shape,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation = "relu"),
        layers.Dense(32, activation = "relu"),
        layers.Dense(3, name = 'bottleneck') # THIS is your x, y, z coordinate (Bottleneck) - add a name for easier location
        
        #this is the decoder part
        layers.Dense(32, activation="relu"),
        layers.Dense(64, activation = "relu"),
        layers.Dense(128, activation = "relu"),
        layers.Dense(input_shape, activation="linear") # MUST be 4 to match [sex, age, pnp, igf]
    ])
    return model

#GO PYTHON FILE NEEDS TO HAVE THIS PART - IT IS HOW YOU RUN THE DEFINITION
input_shape = df_tensor.shape[1] # bc of python indexing - to get the 4 dimension value - you need to chose the 1st column not the 0th
autoencoder = autoenc_build(input_shape)

def compiler():
    autoencoder.compile(optimizer = "Adam",
                        loss = 'mse',
                        metrics = ['accuracy']
                        )
    
def train_autoencoder(autoencoder, epochs = None):
    if epochs is None:
        epochs = mcfg.epoch

    return autoencoder.fit(
        x = df_tensor, 
        y = df_tensor, 
        epochs = mcfg.epoch,
        validation_split = 0.2
    )
    

Train the Encoder

In [ ]:
autoencoder.compile(loss ='mse')
autoencoder.fit(
    x = df,
    y = df,
    validation_split = 0.2,
    epochs = mcfg.epoch
)


Obtaining layer outputs (Middle of the endoer layer outputs) - this is the compressed co-ordiantes for each athlete
autoencoder - 4 in - 3 hidden - 4 out 
The encoder is the top half of the model - this is the 4 in and 3 out 
     - inside the 'latent space vectors' is a new matrix (numpy array)
     - it will have a shape of (99,3)
     - [rows, columns ] - this is the format of co=ordinates in python 


In [ ]:
latent_space_vectors = encoder.predict(df)
#Reattach the volunteer ID df to the df processed by the autoencoder in order to ensure that all are back to being labelled - do not shuffle the data.

print(f"obtained {latent_vectors.shape[0]} latent vectors of dimensions {latent_vectors.shape[1]}")

In [ ]:
#Visualising the hidden co-ordinates 